# ftir_36 — Leakage-safe domain-invariant PLS transfer

## tl;dr

The leakage-safe di-PLS trial does **not** rescue a universal IMPROVE→SPARTAN calibration.
The published/default target-centering convention is structurally wrong for these data:
it forces the target mean predicted *filter loading* toward the IMPROVE mean even though
sites have different loading and sampling-volume distributions. On the locked Delhi
holdout it inflates RMSE from **2.61 to 59.71 µg/m³** for the Addis winner, **2.15 to
28.28** for the common candidate, and **1.93 to 20.09** for the screened Delhi winner.
Target centering with λ=0 already produces nearly the entire failure, so this is not a
regularization-tuning artifact. A source-centered sensitivity retains the frozen di-PLS
weights without that mean shift. It is neutral on Addis and mixed on Delhi: the common
candidate improves Delhi RMSE **9.4%** (2.15→1.95) and R² **0.772→0.836**, with
**0.875x−0.75**; its paired-bootstrap RMSE-change interval only just excludes zero
(**−17.8% to −0.05%**) and does not include model-selection or HIPS-reference uncertainty.
The same candidate remains unusably flat at Addis (**0.424x−0.52**). The Delhi-specific
candidate worsens Delhi RMSE by **19.9%**. On untouched IMPROVE sites source-centered
di-PLS changes RMSE by only about 0 to −2.4%. The covariate-shift premise is violated:
the site difference includes real response/loading-distribution change, not merely a
nuisance spectral-domain shift that latent alignment can remove.

## Context & Methods

This is the preregistered follow-up identified by ftir_35. Domain-invariant PLS
(di-PLS; Nikzad-Langerodi et al., 2018, DOI 10.1021/acs.analchem.8b00498)
aligns source and target latent distributions while fitting responses only in the
labeled source domain.

### Key assumptions and leakage controls

- Three source configurations were frozen before this run: the Addis winner, the
  common two-city candidate, and the screened Delhi winner.
- The ordinary PLS component count is selected only within the 80% IMPROVE source
  training partition using site-grouped CV.
- The domain-penalty multiplier is selected by treating held-out IMPROVE sites as
  unlabeled pseudo-targets. The untouched 20% IMPROVE sites are an outer validation.
- Addis/Delhi spectra may enter as unlabeled target covariates. Their HIPS values are
  not read until the protocol JSON has been written.
- Adjacent channels are averaged in fixed blocks of eight before any fitting. This
  reduces 2,002–2,722 channels to 251–341 features, making the published repeated
  covariance eigendecomposition tractable. Every comparator receives the same bins.
- Final evaluation is on the 14 Addis and 26 Delhi reconstructed filters that were
  never used in the five-site screen. The larger augmented target clouds are used only
  to estimate unlabeled target means/covariances.

In [1]:
%matplotlib inline
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold

sys.path.insert(0, './scripts')

from phase3_common import PATHS, PHASE2_TABLES, load_pool_metadata, load_pool_spectra, load_tor_loadings
from calibration_modes import protocol_train_mask
from domain_invariant_pls import block_average, fit_domain_invariant_pls
from pls_transfer import component_cv_curve, select_first_major_minimum

ROOT = Path('.')
REPO = Path.cwd().parents[1]
TARGETS = REPO / 'calibration_explorer/targets'
OUT = ROOT / 'output/tables/ftir36'
OUT.mkdir(parents=True, exist_ok=True)

BLOCK_SIZE = 8
MAX_COMPONENTS = 20
PSEUDO_TARGET_FOLDS = 3
MULTIPLIERS = np.array([0.0, 0.1, 0.3, 1.0, 3.0])
SAVGOL = dict(window_length=11, polyorder=2, deriv=2)

CONFIGS = {
    'addis_winner': {'cohort': 'ocec', 'cutoff': 440, 'spectra': 'airspec'},
    'common_candidate': {'cohort': 'analogs', 'cutoff': 440, 'spectra': 'deriv2'},
    'delhi_winner': {'cohort': 'analogs', 'cutoff': 530, 'spectra': 'deriv2'},
}
TARGET_SPECS = {
    'addis_holdout': {
        'unlabeled': 'addis_augmented',
        'evaluation': 'addis_reconstructed_holdout',
    },
    'delhi_holdout': {
        'unlabeled': 'indh_augmented',
        'evaluation': 'indh_reconstructed_holdout',
    },
}


def metrics(observed, predicted):
    observed = np.asarray(observed, float)
    predicted = np.asarray(predicted, float)
    slope, intercept = np.polyfit(observed, predicted, 1)
    fitted = slope * observed + intercept
    ss_total = np.sum((predicted - predicted.mean()) ** 2)
    return {
        'slope': float(slope),
        'intercept': float(intercept),
        'R2': float(1 - np.sum((predicted - fitted) ** 2) / ss_total)
        if ss_total > 0 else np.nan,
        'RMSE_1to1': float(np.sqrt(np.mean((predicted - observed) ** 2))),
        'mean_bias': float(np.mean(predicted - observed)),
    }


def bootstrap_rmse_change(observed, baseline, adapted, *, seed, n_boot=5000):
    """Paired percentile interval for percent RMSE change on the locked filters."""
    observed = np.asarray(observed, float)
    baseline = np.asarray(baseline, float)
    adapted = np.asarray(adapted, float)
    rng = np.random.default_rng(seed)
    changes = np.empty(n_boot)
    for index in range(n_boot):
        take = rng.integers(0, len(observed), len(observed))
        base_rmse = np.sqrt(np.mean((baseline[take] - observed[take]) ** 2))
        adapted_rmse = np.sqrt(np.mean((adapted[take] - observed[take]) ** 2))
        changes[index] = 100 * (adapted_rmse / base_rmse - 1)
    return float(np.percentile(changes, 2.5)), float(np.percentile(changes, 97.5))


def load_training_frame():
    metadata = load_pool_metadata().merge(
        load_tor_loadings(), on=['Site', 'date'], how='left', validate='many_to_one')
    pool = metadata.query('TOR_EC_loading_ug > 0').drop_duplicates('FilterId').copy()
    pool['AnalysisId'] = pool['AnalysisId'].astype(int)
    return metadata, pool.drop_duplicates('AnalysisId').set_index('AnalysisId')


def resolve_ids(name, spec, metadata, pool):
    if spec['cohort'] == 'ocec':
        eligible = (metadata['TOR_EC_loading_ug'].gt(0)
                    & metadata['TOR_EC_ugm3'].gt(0)
                    & metadata['TOR_OC_ugm3'].gt(0)
                    & metadata['OC_EC_ratio'].notna())
        ranked = (metadata[eligible].sort_values('OC_EC_ratio')
                  .drop_duplicates('FilterId')['AnalysisId'].astype(int).to_numpy())
    else:
        cached = np.load(REPO / 'calibration_explorer/cache/analog_corrected_ranking.npz')
        ranked = cached['ids'].astype(int)
    ranked = ranked[np.isin(ranked, pool.index.to_numpy())]
    ids = np.array(list(dict.fromkeys(ranked[:spec['cutoff']])), dtype=int)
    if len(ids) != spec['cutoff']:
        raise ValueError(f'{name}: resolved {len(ids)} rather than {spec["cutoff"]} rows')
    return ids


def source_representation(ids, spectra, pool, corrected):
    if spectra == 'airspec':
        row = {int(value): i for i, value in enumerate(corrected['analysis_id'].astype(int))}
        if not set(ids) <= set(row):
            raise ValueError('AIRSpec cache does not cover the resolved cohort')
        X = corrected['corrected'][[row[int(value)] for value in ids]].astype(float)
    else:
        columns = pd.read_csv(
            PATHS.ftir_dir / 'local_db/spectra_248_251.csv', nrows=0).columns
        wcols = sorted(
            [value for value in columns if str(value).replace('.', '', 1).isdigit()],
            key=lambda value: -float(value))
        raw = load_pool_spectra(ids, wcols).set_index('AnalysisId').loc[ids]
        X = savgol_filter(raw[wcols].to_numpy(float), axis=1, **SAVGOL)
    X = block_average(X, BLOCK_SIZE)
    y = pool.loc[ids, 'TOR_EC_loading_ug'].to_numpy(float)
    groups = pool.loc[ids, 'Site'].astype(str).to_numpy()
    return X, y, groups


def load_unlabeled_target(name, spectra):
    folder = TARGETS / name
    path = folder / ('spectra_corrected.csv' if spectra == 'airspec' else 'spectra.csv')
    frame = pd.read_csv(path)
    identifiers = frame.iloc[:, 0].astype(str).to_numpy()
    X = frame.iloc[:, 1:].to_numpy(float)
    if spectra == 'deriv2':
        X = savgol_filter(X, axis=1, **SAVGOL)
    return identifiers, block_average(X, BLOCK_SIZE)


def select_components(X, y, groups, train_mask):
    curve = component_cv_curve(
        X[train_mask], y[train_mask], range(1, MAX_COMPONENTS + 1),
        groups=groups[train_mask], n_splits=5, random_state=42,
    )
    selected, annotated = select_first_major_minimum(curve)
    return int(selected), 'first local minimum within one SE of global minimum', annotated


def select_multiplier(X, y, groups, source_train, n_components):
    positions = np.flatnonzero(source_train)
    split = GroupKFold(n_splits=PSEUDO_TARGET_FOLDS)
    predictions = {float(value): np.full(len(positions), np.nan) for value in MULTIPLIERS}
    for inner_train, pseudo_target in split.split(X[positions], groups=groups[positions]):
        train_pos = positions[inner_train]
        target_pos = positions[pseudo_target]
        heuristic = fit_domain_invariant_pls(
            X[train_pos], y[train_pos], X[target_pos],
            n_components=n_components, heuristic=True,
        )
        for multiplier in MULTIPLIERS:
            fitted = fit_domain_invariant_pls(
                X[train_pos], y[train_pos], X[target_pos],
                n_components=n_components,
                lambdas=heuristic.lambdas * multiplier,
            )
            predictions[float(multiplier)][pseudo_target] = fitted.predict(X[target_pos])
    rows = []
    observed = y[positions]
    for multiplier in MULTIPLIERS:
        predicted = predictions[float(multiplier)]
        if not np.isfinite(predicted).all():
            raise ValueError('pseudo-target predictions are incomplete')
        rows.append({'multiplier': float(multiplier),
                     'pseudo_target_RMSE': float(np.sqrt(np.mean((predicted - observed) ** 2)))})
    table = pd.DataFrame(rows).sort_values(['pseudo_target_RMSE', 'multiplier'])
    return float(table.iloc[0]['multiplier']), table.sort_values('multiplier')


def predict_variants(X_train, y_train, X_domain, X_eval, n_components, multiplier):
    ordinary = PLSRegression(n_components=n_components, scale=False).fit(X_train, y_train)
    zero = fit_domain_invariant_pls(
        X_train, y_train, X_domain, n_components=n_components, lambdas=0)
    heuristic = fit_domain_invariant_pls(
        X_train, y_train, X_domain, n_components=n_components, heuristic=True)
    selected = fit_domain_invariant_pls(
        X_train, y_train, X_domain, n_components=n_components,
        lambdas=heuristic.lambdas * multiplier,
    )
    return {
        'ordinary_pls': ordinary.predict(X_eval).ravel(),
        'target_centered_lambda0': zero.predict(X_eval),
        'dipls_target_centered_heuristic': heuristic.predict(X_eval),
        'dipls_target_centered_selected': selected.predict(X_eval),
        'dipls_source_centered_heuristic': heuristic.predict(
            X_eval, centering='source'),
        'dipls_source_centered_selected': selected.predict(
            X_eval, centering='source'),
    }, heuristic.lambdas

## Source-only selection

No target response file has been opened at this point.

In [2]:
metadata, pool = load_training_frame()
corrected = np.load(ROOT / 'output/corrected/improve_pool_corrected_df6.npz', allow_pickle=True)
source_data = {}
selection_rows = []
lambda_curves = []

for config_name, spec in CONFIGS.items():
    ids = resolve_ids(config_name, spec, metadata, pool)
    X, y, groups = source_representation(ids, spec['spectra'], pool, corrected)
    source_train = protocol_train_mask('site_heldout', X, y, groups)
    n_components, component_reason, component_curve = select_components(
        X, y, groups, source_train)
    multiplier, multiplier_curve = select_multiplier(
        X, y, groups, source_train, n_components)
    source_data[config_name] = {
        'spec': spec, 'ids': ids, 'X': X, 'y': y, 'groups': groups,
        'source_train': source_train, 'n_components': n_components,
        'multiplier': multiplier,
    }
    selection_rows.append({
        'config': config_name, **spec, 'n_source': len(ids),
        'n_source_train': int(source_train.sum()),
        'n_source_outer_holdout': int((~source_train).sum()),
        'n_source_sites': int(pd.Series(groups).nunique()),
        'n_features_binned': X.shape[1], 'n_components': n_components,
        'component_rule': component_reason,
        'selected_multiplier': multiplier,
    })
    multiplier_curve.insert(0, 'config', config_name)
    lambda_curves.append(multiplier_curve)
    component_curve.assign(config=config_name).to_csv(
        OUT / f'{config_name}_component_curve.csv', index=False)

selection = pd.DataFrame(selection_rows)
lambda_selection = pd.concat(lambda_curves, ignore_index=True)
selection.to_csv(OUT / 'frozen_protocol.csv', index=False)
lambda_selection.to_csv(OUT / 'lambda_selection_curves.csv', index=False)
(OUT / 'frozen_protocol.json').write_text(json.dumps({
    'block_size': BLOCK_SIZE,
    'max_components': MAX_COMPONENTS,
    'pseudo_target_folds': PSEUDO_TARGET_FOLDS,
    'candidate_multipliers': MULTIPLIERS.tolist(),
    'target_labels_opened': False,
    'configurations': selection.to_dict(orient='records'),
}, indent=2))

display(selection)
display(lambda_selection.pivot(index='multiplier', columns='config', values='pseudo_target_RMSE').round(3))

,config,cohort,cutoff,spectra,n_source,n_source_train,n_source_outer_holdout,n_source_sites,n_features_binned,n_components,component_rule,selected_multiplier
0,addis_winner,ocec,440,airspec,440,387,53,105,251,5,first local minimum within one SE of global mi...,0.1
1,common_candidate,analogs,440,deriv2,440,334,106,116,341,15,first local minimum within one SE of global mi...,1.0
2,delhi_winner,analogs,530,deriv2,530,423,107,127,341,12,first local minimum within one SE of global mi...,1.0


config,addis_winner,common_candidate,delhi_winner
multiplier,,,
0.0,8.157,1.628,1.510
0.1,8.125,1.626,1.513
0.3,8.132,1.621,1.504
1.0,8.294,1.615,1.497
3.0,8.613,1.619,1.516


## Untouched IMPROVE-site validation

This is the first use of the 20% outer source holdout. It checks whether the chosen
domain penalty improves transfer between unseen IMPROVE sites before any Addis/Delhi
label is opened.

In [3]:
source_validation_rows = []
for config_name, data in source_data.items():
    train = data['source_train']
    variants, heuristic_lambdas = predict_variants(
        data['X'][train], data['y'][train], data['X'][~train], data['X'][~train],
        data['n_components'], data['multiplier'])
    for variant, predicted in variants.items():
        source_validation_rows.append({
            'config': config_name, 'variant': variant,
            'n': int((~train).sum()), **metrics(data['y'][~train], predicted),
            'median_heuristic_lambda': float(np.median(heuristic_lambdas)),
        })

source_validation = pd.DataFrame(source_validation_rows)
source_validation.to_csv(OUT / 'outer_source_validation.csv', index=False)
display(source_validation.round(3))

,config,variant,n,slope,intercept,R2,RMSE_1to1,mean_bias,median_heuristic_lambda
0,addis_winner,ordinary_pls,53,0.957,-0.179,0.908,3.714,-0.466,3.736566e+08
1,addis_winner,target_centered_lambda0,53,0.957,-0.959,0.908,3.890,-1.246,3.736566e+08
2,addis_winner,dipls_target_centered_heuristic,53,0.977,-1.094,0.907,3.948,-1.246,3.736566e+08
3,addis_winner,dipls_target_centered_selected,53,0.958,-0.967,0.908,3.893,-1.246,3.736566e+08
4,addis_winner,dipls_source_centered_heuristic,53,0.977,-0.353,0.907,3.780,-0.505,3.736566e+08
5,addis_winner,dipls_source_centered_selected,53,0.958,-0.191,0.908,3.718,-0.471,3.736566e+08
6,common_candidate,ordinary_pls,106,0.756,1.119,0.842,1.412,0.040,1.548176e+12
7,common_candidate,target_centered_lambda0,106,0.756,0.336,0.842,1.595,-0.743,1.548176e+12
8,common_candidate,dipls_target_centered_heuristic,106,0.775,0.254,0.843,1.580,-0.743,1.548176e+12
9,common_candidate,dipls_target_centered_selected,106,0.775,0.254,0.843,1.580,-0.743,1.548176e+12


## Locked target reveal

The protocol is now frozen on disk. The following cell is the first code that reads an
Addis or Delhi `reference.csv`. Predictions use HIPS only for final scoring.

In [4]:
target_summary_rows = []
target_prediction_rows = []

for config_name, data in source_data.items():
    train = data['source_train']
    spectra_kind = data['spec']['spectra']
    for target_name, target_spec in TARGET_SPECS.items():
        _, X_domain = load_unlabeled_target(target_spec['unlabeled'], spectra_kind)
        eval_ids, X_eval = load_unlabeled_target(target_spec['evaluation'], spectra_kind)
        reference = pd.read_csv(TARGETS / target_spec['evaluation'] / 'reference.csv')
        id_column = reference.columns[0]
        reference[id_column] = reference[id_column].astype(str)
        reference = reference.set_index(id_column).loc[eval_ids].reset_index()
        if not (reference['Volume_m3'] > 0).all():
            raise ValueError(f'{target_name}: invalid sample volume')
        observed = reference['Fabs'].to_numpy(float) / 10.0
        variants, heuristic_lambdas = predict_variants(
            data['X'][train], data['y'][train], X_domain, X_eval,
            data['n_components'], data['multiplier'])
        for variant, predicted_loading in variants.items():
            predicted = predicted_loading / reference['Volume_m3'].to_numpy(float)
            target_summary_rows.append({
                'config': config_name, 'target': target_name, 'variant': variant,
                'n': len(reference), 'n_unlabeled_domain': len(X_domain),
                'n_components': data['n_components'],
                'selected_multiplier': data['multiplier'],
                'median_heuristic_lambda': float(np.median(heuristic_lambdas)),
                **metrics(observed, predicted),
            })
            for filter_id, obs, pred in zip(eval_ids, observed, predicted):
                target_prediction_rows.append({
                    'config': config_name, 'target': target_name, 'variant': variant,
                    'ExternalFilterId': filter_id, 'observed_bc_mac10_ugm3': obs,
                    'predicted_ec_ugm3': pred, 'residual_ugm3': pred - obs,
                })

target_summary = pd.DataFrame(target_summary_rows)
target_predictions = pd.DataFrame(target_prediction_rows)
target_summary.to_csv(OUT / 'locked_target_summary.csv', index=False)
target_predictions.to_csv(OUT / 'locked_target_predictions.csv', index=False)
(OUT / 'run_manifest.json').write_text(json.dumps({
    'block_size': BLOCK_SIZE,
    'max_components': MAX_COMPONENTS,
    'pseudo_target_folds': PSEUDO_TARGET_FOLDS,
    'candidate_multipliers': MULTIPLIERS.tolist(),
    'target_labels_opened': True,
    'frozen_protocol': 'frozen_protocol.json',
    'configurations': selection.to_dict(orient='records'),
}, indent=2))

display(target_summary.round(3))

,config,target,variant,n,n_unlabeled_domain,n_components,selected_multiplier,median_heuristic_lambda,slope,intercept,R2,RMSE_1to1,mean_bias
0,addis_winner,addis_holdout,ordinary_pls,14,253,5,0.1,1.866113e+08,0.459,-0.417,0.769,2.812,-2.777
1,addis_winner,addis_holdout,target_centered_lambda0,14,253,5,0.1,1.866113e+08,0.457,-2.167,0.752,4.561,-4.539
2,addis_winner,addis_holdout,dipls_target_centered_heuristic,14,253,5,0.1,1.866113e+08,0.500,-2.317,0.776,4.520,-4.500
3,addis_winner,addis_holdout,dipls_target_centered_selected,14,253,5,0.1,1.866113e+08,0.461,-2.182,0.756,4.555,-4.533
4,addis_winner,addis_holdout,dipls_source_centered_heuristic,14,253,5,0.1,1.866113e+08,0.502,-0.480,0.792,2.684,-2.652
5,addis_winner,addis_holdout,dipls_source_centered_selected,14,253,5,0.1,1.866113e+08,0.464,-0.423,0.772,2.797,-2.763
6,addis_winner,delhi_holdout,ordinary_pls,26,178,5,0.1,2.256294e+08,1.069,0.815,0.590,2.607,1.314
7,addis_winner,delhi_holdout,target_centered_lambda0,26,178,5,0.1,2.256294e+08,4.490,-50.296,0.043,59.539,-24.888
8,addis_winner,delhi_holdout,dipls_target_centered_heuristic,26,178,5,0.1,2.256294e+08,4.472,-49.770,0.044,58.895,-24.494
9,addis_winner,delhi_holdout,dipls_target_centered_selected,26,178,5,0.1,2.256294e+08,4.504,-50.433,0.043,59.708,-24.927


## Results

The comparison below isolates three effects: ordinary binned PLS, target-mean centering
without a covariance penalty (`lambda0`), and the actual domain-invariant penalty. The
heuristic row is the paper/library default; the selected row uses only the pseudo-target
source-site rule above. Source-centering is a controlled sensitivity using the same frozen
weights and multiplier; it tests whether covariance alignment helps after disabling the
target-mean shift that proved inappropriate for filter-loading responses.

In [5]:
pivot = target_summary.pivot_table(
    index=['config', 'target'], columns='variant',
    values=['slope', 'intercept', 'R2', 'RMSE_1to1', 'mean_bias'])
display(pivot.round(3))

improvements = []
for group_index, ((config_name, target_name), group) in enumerate(
        target_summary.groupby(['config', 'target'])):
    ordinary = group.set_index('variant').loc['ordinary_pls']
    indexed = group.set_index('variant')
    for centering, variant in (
        ('target', 'dipls_target_centered_selected'),
        ('source', 'dipls_source_centered_selected'),
    ):
        selected = indexed.loc[variant]
        paired = target_predictions[
            target_predictions['config'].eq(config_name)
            & target_predictions['target'].eq(target_name)
            & target_predictions['variant'].isin(['ordinary_pls', variant])
        ].pivot(index='ExternalFilterId', columns='variant',
                values=['observed_bc_mac10_ugm3', 'predicted_ec_ugm3'])
        observed = paired['observed_bc_mac10_ugm3']['ordinary_pls'].to_numpy(float)
        baseline_prediction = paired['predicted_ec_ugm3']['ordinary_pls'].to_numpy(float)
        adapted_prediction = paired['predicted_ec_ugm3'][variant].to_numpy(float)
        change_low, change_high = bootstrap_rmse_change(
            observed, baseline_prediction, adapted_prediction,
            seed=20260824 + 10 * group_index + (centering == 'source'))
        improvements.append({
            'config': config_name, 'target': target_name, 'centering': centering,
            'selected_multiplier': selected['selected_multiplier'],
            'ordinary_RMSE': ordinary['RMSE_1to1'],
            'dipls_RMSE': selected['RMSE_1to1'],
            'RMSE_change_pct': 100 * (
                selected['RMSE_1to1'] / ordinary['RMSE_1to1'] - 1),
            'RMSE_change_ci_low': change_low,
            'RMSE_change_ci_high': change_high,
            'ordinary_slope': ordinary['slope'], 'dipls_slope': selected['slope'],
            'ordinary_intercept': ordinary['intercept'],
            'dipls_intercept': selected['intercept'],
        })
improvements = pd.DataFrame(improvements)
improvements.to_csv(OUT / 'dipls_vs_pls.csv', index=False)
display(improvements.round(3))

R2  \
variant                        dipls_source_centered_heuristic   
config           target                                          
addis_winner     addis_holdout                           0.792   
                 delhi_holdout                           0.513   
common_candidate addis_holdout                           0.818   
                 delhi_holdout                           0.836   
delhi_winner     addis_holdout                           0.703   
                 delhi_holdout                           0.443   

                                                               \
variant                        dipls_source_centered_selected   
config           target                                         
addis_winner     addis_holdout                          0.772   
                 delhi_holdout                          0.584   
common_candidate addis_holdout                          0.818   
                 delhi_holdout                          0.836   
delhi_winner     addis_holdout                          0.703   
                 delhi_holdout                          0.443   

                                                                \
variant                        dipls_target_centered_heuristic   
config           target                                          
addis_winner     addis_holdout                           0.776   
                 delhi_holdout                           0.044   
common_candidate addis_holdout                           0.819   
                 delhi_holdout                           0.056   
delhi_winner     addis_holdout                           0.702   
                 delhi_holdout                           0.075   

                                                                            \
variant                        dipls_target_centered_selected ordinary_pls   
config           target                                                      
addis_winner     addis_holdout                          0.756        0.769   
                 delhi_holdout                          0.043        0.590   
common_candidate addis_holdout                          0.819        0.783   
                 delhi_holdout                          0.056        0.772   
delhi_winner     addis_holdout                          0.702        0.782   
                 delhi_holdout                          0.075        0.573   

                                                        \
variant                        target_centered_lambda0   
config           target                                  
addis_winner     addis_holdout                   0.752   
                 delhi_holdout                   0.043   
common_candidate addis_holdout                   0.785   
                 delhi_holdout                   0.055   
delhi_winner     addis_holdout                   0.781   
                 delhi_holdout                   0.063   

                                                     RMSE_1to1  \
variant                        dipls_source_centered_heuristic   
config           target                                          
addis_winner     addis_holdout                           2.684   
                 delhi_holdout                           3.108   
common_candidate addis_holdout                           3.069   
                 delhi_holdout                           1.951   
delhi_winner     addis_holdout                           2.916   
                 delhi_holdout                           2.317   

                                                               \
variant                        dipls_source_centered_selected   
config           target                                         
addis_winner     addis_holdout                          2.797   
                 delhi_holdout                          2.663   
common_candidate addis_holdout                          3.069   
                 delhi_holdout                          1.951   
delhi_wi

,config,target,centering,selected_multiplier,ordinary_RMSE,dipls_RMSE,RMSE_change_pct,RMSE_change_ci_low,RMSE_change_ci_high,ordinary_slope,dipls_slope,ordinary_intercept,dipls_intercept
0,addis_winner,addis_holdout,target,0.1,2.812,4.555,61.962,57.753,67.461,0.459,0.461,-0.417,-2.182
1,addis_winner,addis_holdout,source,0.1,2.812,2.797,-0.537,-0.584,-0.496,0.459,0.464,-0.417,-0.423
2,addis_winner,delhi_holdout,target,0.1,2.607,59.708,2190.508,403.006,3312.845,1.069,4.504,0.815,-50.433
3,addis_winner,delhi_holdout,source,0.1,2.607,2.663,2.162,0.749,3.564,1.069,1.068,0.815,0.882
4,common_candidate,addis_holdout,target,1.0,3.185,4.292,34.748,32.075,37.761,0.422,0.422,-0.630,-1.745
5,common_candidate,addis_holdout,source,1.0,3.185,3.069,-3.657,-4.257,-3.090,0.422,0.424,-0.630,-0.520
6,common_candidate,delhi_holdout,target,1.0,2.155,28.276,1212.199,286.744,1843.006,0.920,2.389,-1.153,-23.360
7,common_candidate,delhi_holdout,source,1.0,2.155,1.951,-9.441,-17.808,-0.054,0.920,0.875,-1.153,-0.748
8,delhi_winner,addis_holdout,target,1.0,2.943,3.970,34.878,32.156,38.014,0.377,0.367,-0.185,-1.176
9,delhi_winner,addis_holdout,source,1.0,2.943,2.916,-0.938,-1.704,-0.194,0.377,0.369,-0.185,-0.117


## Takeaways

- **Do not use default target-centered di-PLS for filter-loading calibration.** It erases
  a physically meaningful target loading mean, and division by site-specific sample volume
  amplifies the resulting error.
- **The failure is not label leakage or a bad target-selected λ.** Components and the
  penalty multiplier were selected entirely within IMPROVE; the frozen protocol was written
  before the 14 Addis and 26 Delhi HIPS values were opened.
- **Covariance alignment alone offers no general solution.** Source-centered weights give
  one modest Delhi improvement, but it is configuration-specific and the same model retains
  a ~0.42 Addis slope.
- **Keep ordinary PLS as the calibration baseline.** Further adaptation would need an
  explicitly conditional/response-aware method and genuinely labeled target anchors; at
  that point it is local calibration, not unsupervised domain adaptation.